In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_DIR / "src"

sys.path.append(str(SRC_DIR))
print(PROJECT_DIR)

In [ ]:
from core.config import update_path_settings

update_path_settings(PROJECT_DIR)

In [ ]:
from core.setup import settings, setup_data

DOWNLOAD_DATA = False
if DOWNLOAD_DATA:
    setup_data()

In [ ]:
SPECIES_DIRS = [d for d in settings.DATA_RAW_DIR.iterdir() if d.is_dir()]
SPECIES_DIRS.sort()

LW_SPECIES_DIR = [d for d in SPECIES_DIRS if "LW" in d.name][0]

In [ ]:
from domain.pipelines.annotations import (
    append_metrics,
    df_to_annotations,
    filter_noise,
    load_annotation_file,
    normalize_header,
    normalize_species_and_calls,
    parse_numerics,
)
from domain.pipelines.audio import load_audio_torchaudio
from domain.pipelines.types import Annotation, AudioRecord

ANNOTATIONS_EXT = ".txt"
AUDIO_EXT = ".wav"
SAMPLE_RATE = 44100

recordings: list[AudioRecord] = []
for file in LW_SPECIES_DIR.iterdir():
    if file.suffix.lower() != AUDIO_EXT:
        continue

    audio_path = file
    annotation_df_path = file.with_suffix(ANNOTATIONS_EXT)
    annotations: list[Annotation] = []

    if annotation_df_path.exists():
        df = load_annotation_file(annotation_df_path)
        df = normalize_header(df)
        df = normalize_species_and_calls(df)
        df = parse_numerics(df)
        df = filter_noise(df)
        df = append_metrics(df)
        annotations = df_to_annotations(df)

    wav_tensor = load_audio_torchaudio(audio_path, sample_rate=SAMPLE_RATE)
    record = AudioRecord(
        wav=wav_tensor,
        sample_rate=SAMPLE_RATE,
        annotations=annotations,
    )
    recordings.append(record)


In [ ]:
import matplotlib.pyplot as plt

from domain.pipelines.annotations import annotations_to_df
from domain.pipelines.image import compute_spectrogram

NFFT = 2048
HOP = 512
sample_record = recordings[10]
spec = compute_spectrogram(sample_record.wav, n_fft=NFFT, hop_length=HOP)
plt.figure(figsize=(20, 4))
plt.imshow(spec[0].numpy(), aspect="auto", origin="lower")
plt.title(f"Spectrogram {sample_record.sample_rate}Hz")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
for ann in sample_record.annotations:
    x1 = int(ann.begin_time * sample_record.sample_rate / HOP)
    x2 = int(ann.end_time * sample_record.sample_rate / HOP)
    y1 = int(ann.low_freq * NFFT / sample_record.sample_rate)
    y2 = int(ann.high_freq * NFFT / sample_record.sample_rate)
    plt.gca().add_patch(
        plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )
    )

plt.show()

df = annotations_to_df(sample_record.annotations)
df

In [ ]:
from domain.pipelines.pipeline import SpectrogramDataset, WindowConfig

window_cfg = WindowConfig(
    duration_sec=3.0,
    hop_sec=1.5,
    n_fft=2048,
    hop_length=512,
    sample_rate=SAMPLE_RATE,
    scale_method="min_max",
    overlap_threshold=0.5,
    img_size=640,
    n_channels=3,
)


def class_mapping_fn(species: str, call_type: str) -> int:
    mapping = {
        "cs": 0,  # call_syllable
        "cc": 1,  # call_cluster/phrase
    }
    return mapping.get(call_type, 2)


In [ ]:
from sklearn.model_selection import train_test_split

from domain.pipelines.pipeline import export_to_yolo, write_data_yaml

train_recs, val_recs = train_test_split(
    recordings,
    test_size=0.2,
    random_state=42,
)

ds_train = SpectrogramDataset(
    recordings=train_recs, cfg=window_cfg, class_mapping_fn=class_mapping_fn
)

ds_val = SpectrogramDataset(recordings=val_recs, cfg=window_cfg, class_mapping_fn=class_mapping_fn)

YOLO_PATH = settings.DATA_DIR / "yolo"

export_to_yolo(ds_train, YOLO_PATH, split="train")
export_to_yolo(ds_val, YOLO_PATH, split="val")


In [ ]:
write_data_yaml(
    output_path=YOLO_PATH,
    train_split="train",
    val_split="val",
    names={
        0: "call_syllable",
        1: "call_cluster",
        2: "other",
    },
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data=YOLO_PATH / "data.yaml",
    epochs=10,
    imgsz=640,
    augment=True,
    fliplr=0.0,
)